# Dear Canada — Single-book Topic Model (themes over time)

**Question this notebook answers:** *what is this book about, and how do its themes rise, fall and hand over to each other as the diary unfolds?*

**Input:** one CSV in `./data/` named `{BOOK_CODE}.csv` with columns `book, chapter_id, entry_id, date, entry_text` (the same file your KG pipeline reads). Text is assumed to be raw and uncleaned.

**Method.** Clean → lemmatise and drop people's names (spaCy) → merge consecutive short entries into *segments* (never across chapters) → TF-IDF → NMF topic model, K chosen by NPMI coherence → score **every individual entry** against the fitted topics → analyse the topic weights along two time axes:

- **narrative time** — entry order, always available;
- **calendar time** — the dates in the diary, where they can be read. Diaries compress and skip time (a gap of weeks may be one entry), so the two views can look quite different and both are worth looking at.

This is not a "dynamic topic model": topics are fitted once on the whole book and then *tracked* through it. On a book-sized corpus that is more stable and more honest than refitting topics on tiny time slices.

**Outputs** (in `./output/topics_book/{BOOK_CODE}/`): `entry_topics.csv` (per-entry weights, joins to the KG on `entry_id`), `topic_summary.csv`, `topic_terms.csv`, `chapter_topics.csv`, `topic_shifts.csv`, `exemplars.csv`, and interactive HTML figures in `figures/`.

**Requires:** `pip install pandas numpy scipy scikit-learn spacy plotly tqdm` and `python -m spacy download en_core_web_sm`. The optional Claude labelling step also needs `anthropic` and `python-dotenv`.

## 0. Setup & configuration

In [1]:
import os
import re
import json
import html
import hashlib
import unicodedata
import warnings
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import sparse
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist, jensenshannon
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from scipy.signal import find_peaks

In [2]:
# --- which book ----------------------------------------------------------------
BOOK_CODE = "SofS_entries_halved"           # filename stem in ./data, same as the KG notebook
DATA_DIR = Path("./data")
OUT_DIR = Path("./output/topics_book") / BOOK_CODE
FIG_DIR = OUT_DIR / "figures"
CACHE_DIR = OUT_DIR / "cache"
for d in (OUT_DIR, FIG_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
BOOK_TITLE = None                           # optional display name, e.g. "A Sea of Sorrows"
START_YEAR = None                           # only if NO entry date contains a year, e.g. 1847

# --- text preprocessing ----------------------------------------------------------
SPACY_MODEL = "en_core_web_sm"
REBUILD_LEMMA_CACHE = False
EXTRA_STOPWORDS = set()                     # add words that clutter topics, e.g. {"ship"} if every topic is "ship ..."

# --- segmenting & vocabulary -------------------------------------------------------
MIN_SEGMENT_TOKENS = 80                     # content words per segment (a single book is short; keep this smaller than in the corpus model)
NGRAM_RANGE = (1, 2)
MIN_DF = 2                                  # a term must appear in >= this many segments
MAX_DF = 0.50                               # ...and in fewer than this share of segments

# --- topic model -------------------------------------------------------------------
K_RANGE = range(3, 13)
K_FINAL = None                              # None = best coherence; or set an int after viewing the sweep
COHERENCE_TOPN = 10
MIN_SEGS_PER_TOPIC = 4                      # caps K so each topic has >= this many segments' worth of data (stops tiny books picking K=12)

# --- temporal analysis ---------------------------------------------------------------
SMOOTH_WINDOW = 5                           # entries in the rolling mean used for the stacked-area / ribbon plots
SHIFT_WINDOW = 4                            # entries on each side compared when looking for topic shifts
N_SHIFTS = 8                                # how many biggest shifts to report

# --- optional Claude labelling -------------------------------------------------------
USE_LLM_LABELS = False
MODEL_NAME = "claude-sonnet-5-5"            # match your KG notebook's MODEL_NAME
CONTEXT_NOTE = ""                           # optional 1-2 sentence synopsis of the book to help the labeller
                                            # (like BASIC_CONTEXT in the KG notebook). Keep it factual and spoiler-light.

## 1. Load the book

Entries are put in reading order using the same chapter/entry-number parsing as `load_book` in the KG notebook.

In [3]:
REQUIRED = ["book", "chapter_id", "entry_id", "date", "entry_text"]

def load_book(book_code, data_dir=DATA_DIR):
    path = data_dir / f"{book_code}.csv"
    df = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, keep_default_na=False)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise RuntimeError(f"Could not decode {path}")
    missing = set(REQUIRED) - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    df = df[REQUIRED].copy()
    df["book"] = df["book"].str.strip().replace("", book_code)
    df["file_order"] = np.arange(len(df))
    df["uid"] = df["book"] + "|" + df["entry_id"]
    df["_ch"] = pd.to_numeric(df["chapter_id"].str.extract(r"(\d+)")[0], errors="coerce").fillna(0)
    df["_en"] = pd.to_numeric(df["entry_id"].str.extract(r"e(\d+)$")[0], errors="coerce").fillna(df["file_order"])
    n0 = len(df)
    df = df.drop_duplicates("uid", keep="first")
    if len(df) < n0:
        print(f"dropped {n0 - len(df)} duplicate entry_id rows")
    df = df.sort_values(["_ch", "_en", "file_order"]).reset_index(drop=True)
    df["pos"] = np.arange(len(df))
    if df["book"].nunique() > 1:
        print("!! This file contains several book values:", df["book"].unique().tolist(), "- timeline plots assume ONE book.")
    return df

book_df = load_book(BOOK_CODE)
TITLE = BOOK_TITLE or BOOK_CODE
print(f"{TITLE}: {len(book_df)} entries in {book_df['chapter_id'].nunique()} chapters")
if len(book_df) < 30:
    print("!! Fewer than ~30 entries: topics will be unstable. Treat results as exploratory and use a small K.")

SofS_entries_halved: 69 entries in 5 chapters


## 2. Clean the raw text

In [4]:
# Raw CSV text is NOT assumed to be clean. This function removes the usual
# artefacts of scanned / converted book text *before* any linguistic processing.
# It is deliberately conservative: it never deletes words, only formatting noise.

QUOTES = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": " - ",
})

def clean_text(text, date_raw=""):
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = re.sub(r"<[^>]+>", " ", t)                          # stray HTML / XML tags
    t = unicodedata.normalize("NFKC", t)                    # ligatures (ﬁ -> fi), odd spaces
    t = t.replace("\u00ad", "")                             # soft hyphens
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)            # URLs
    t = re.sub(r"(\w)-[ \t]*\r?\n[ \t]*(\w)", r"\1\2", t)   # words hyphenated across line breaks
    t = t.translate(QUOTES)                                 # curly quotes / dashes -> ASCII
    t = re.sub(r"\[\d{1,3}\]", " ", t)                      # footnote markers like [12]
    t = re.sub(r"(?im)^[ \t]*(page[ \t]+)?\d{1,4}[ \t]*$", " ", t)   # lines that are only a page number
    t = re.sub(r"\s+", " ", t).strip()                      # collapse all whitespace
    d = str(date_raw).strip()
    if d and t.lower().startswith(d.lower()):               # date heading repeated inside the text
        t = t[len(d):].lstrip(" .,:;-")
    return t

In [5]:
book_df["text_clean"] = [clean_text(t, d) for t, d in zip(book_df["entry_text"], book_df["date"])]
empty = book_df["text_clean"].str.len() == 0
if empty.any():
    print(f"Dropping {int(empty.sum())} entries with no text after cleaning")
    book_df = book_df[~empty].reset_index(drop=True)
    book_df["pos"] = np.arange(len(book_df))

i = book_df.index[len(book_df) // 2]
print("RAW  :", repr(book_df.loc[i, "entry_text"][:300]))
print("CLEAN:", repr(book_df.loc[i, "text_clean"][:300]))

RAW  : 'The ship is moving so slowly that a man ashore could surely make faster progress. But we are in Canada, and from the ship can see hills and woods and neat little farmhouses at the river’s edge.'
CLEAN: "The ship is moving so slowly that a man ashore could surely make faster progress. But we are in Canada, and from the ship can see hills and woods and neat little farmhouses at the river's edge."


## 3. Dates

In [6]:
# Dates are messy in diary books ("Tuesday, May 4" has no year). Strategy:
#   1. read year / month / day out of each raw date string (ISO or free text);
#   2. carry the most recent explicit year forward within a book;
#   3. if the month number wraps backwards by 6+ months (Dec -> Jan), add a year;
#   4. flag every year that was inferred rather than read, so you can audit it.
# If a book has no year anywhere, add it to BOOK_START_YEAR in the config cell.

MONTH_NAMES = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12,
}
_month_re = re.compile(r"\b(" + "|".join(sorted(MONTH_NAMES, key=len, reverse=True)) + r")\b\.?", re.I)
_year_re = re.compile(r"\b(1[5-9]\d{2}|20[0-2]\d)\b")
_iso_re = re.compile(r"\b(\d{4})-(\d{1,2})(?:-(\d{1,2}))?\b")

def parse_date_parts(raw):
    s = str(raw or "").strip()
    if not s:
        return (None, None, None)
    m = _iso_re.search(s)
    if m and 1 <= int(m.group(2)) <= 12:
        return int(m.group(1)), int(m.group(2)), (int(m.group(3)) if m.group(3) else None)
    y = _year_re.search(s)
    mo = _month_re.search(s)
    year = int(y.group(1)) if y else None
    month = MONTH_NAMES[mo.group(1).lower()] if mo else None
    day = None
    rest = _year_re.sub(" ", s)
    dm = re.search(r"\b(\d{1,2})(?:st|nd|rd|th)?\b", rest)
    if dm and 1 <= int(dm.group(1)) <= 31:
        day = int(dm.group(1))
    return year, month, day

def resolve_book_dates(g, start_year=None):
    parts = [parse_date_parts(x) for x in g["date"]]
    explicit = [p[0] for p in parts if p[0]]
    cur = explicit[0] if explicit else start_year
    years, inferred, prev_month = [], [], None
    for y, m, d in parts:
        if y:
            cur, flag = y, False
        else:
            flag = cur is not None
            if cur is not None and m and prev_month and m < prev_month and (prev_month - m) >= 6:
                cur += 1
        if m:
            prev_month = m
        years.append(cur if cur is not None else np.nan)
        inferred.append(flag)
    return pd.DataFrame({
        "year": years,
        "month": pd.to_numeric([p[1] for p in parts], errors="coerce"),
        "day": pd.to_numeric([p[2] for p in parts], errors="coerce"),
        "year_inferred": inferred,
    }, index=g.index)

In [7]:
book_df = book_df.join(resolve_book_dates(book_df, START_YEAR))
book_df["ym"] = pd.Series(
    [pd.Period(year=int(y), month=int(m), freq="M") if not (pd.isna(y) or pd.isna(m)) else pd.NaT
     for y, m in zip(book_df["year"], book_df["month"])], dtype="period[M]")
n_dated = int(book_df["ym"].notna().sum())
print(f"{n_dated}/{len(book_df)} entries have a readable year+month "
      f"({book_df['year_inferred'].mean():.0%} of years were inferred, not read)")
if n_dated:
    print(f"Date span: {book_df['ym'].min()} to {book_df['ym'].max()}")
book_df[["entry_id", "chapter_id", "date", "year", "month", "day", "year_inferred"]].head(8)

69/69 entries have a readable year+month (0% of years were inferred, not read)
Date span: 1847-04 to 1847-08


,entry_id,chapter_id,date,year,month,day,year_inferred
0,chc01_e1,c01,"April 30, 1847",1847,4,30,False
1,chc02_e1,c02,"May 1, 1847",1847,5,1,False
2,chc02_e2,c02,"May 3, 1847",1847,5,3,False
3,chc02_e3,c02,"May 6, 1847",1847,5,6,False
4,chc02_e4,c02,"May 7, 1847",1847,5,7,False
5,chc02_e5,c02,"May 8, 1847",1847,5,8,False
6,chc02_e6,c02,"May 9, 1847",1847,5,9,False
7,chc02_e7,c02,"May 10, 1847",1847,5,10,False


## 4. Lemmatise & filter to content words

Cached in `output/topics_book/<book>/cache/` so re-runs are instant.

In [8]:
# Linguistic preprocessing: lemmatise, keep content words, drop people's names.
#
# Why names are removed: in a diary novel the most frequent "words" are the
# narrator's family. Left in, every topic becomes "a book's cast list" rather
# than a theme. Place names and group names (GPE/LOC/FAC/NORP/EVENT) ARE kept,
# because they carry the spatial and cultural/ethnic signal this project cares about.

KEEP_POS = {"NOUN", "VERB", "ADJ"}
KEEP_ENTS = {"GPE", "LOC", "FAC", "NORP", "EVENT"}

MONTHS_DAYS = set("""january february march april may june july august september october november
december jan feb mar apr jun jul aug sep sept oct nov dec monday tuesday wednesday thursday
friday saturday sunday mon tue tues wed thu thur thurs fri sat sun""".split())

GENERIC = set("""dear diary today tomorrow yesterday tonight morning evening afternoon night day
week month year time thing way say go come get make know think see look want take give tell ask
let put seem feel keep try find little long good great old new big first last next other own able
sure right oh yes well mr mrs miss ms sir madam like just really very much many one two three
also even still back around away again always never ever already though""".split())

STOPWORDS = MONTHS_DAYS | GENERIC | set(EXTRA_STOPWORDS)

def load_nlp(name):
    try:
        import spacy
        nlp = spacy.load(name, disable=["parser"])
        nlp.max_length = 2_000_000
        print(f"Loaded spaCy model: {name}")
        return nlp
    except Exception as exc:
        print(f"!! Could not load spaCy model {name!r} ({exc}).")
        print("   Falling back to plain regex tokenisation: no lemmas, no name removal.")
        print(f"   Fix with:  python -m spacy download {name}")
        return None

def lemmatise_texts(texts, nlp, batch_size=32, n_process=1):
    out = []
    if nlp is None:
        for t in texts:
            toks = [w for w in re.findall(r"[a-z]{3,}", t.lower()) if w not in STOPWORDS]
            out.append(" ".join(toks))
        return out
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size, n_process=n_process), total=len(texts), desc="spaCy"):
        toks = []
        for tok in doc:
            if tok.ent_type_ == "PERSON":
                continue
            if not (tok.pos_ in KEEP_POS or tok.ent_type_ in KEEP_ENTS):
                continue
            if not tok.is_alpha or tok.is_stop:
                continue
            lem = tok.lemma_.lower()
            if len(lem) < 3 or lem in STOPWORDS:
                continue
            toks.append(lem)
        out.append(" ".join(toks))
    return out

def add_lemmas(df, nlp, cache_path, rebuild=False):
    # Incremental cache: only entries whose cleaned text is new or changed are re-processed,
    # so adding a book later does not re-run spaCy over the whole corpus.
    df = df.copy()
    df["text_hash"] = df["text_clean"].map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()[:12])
    if cache_path.exists() and not rebuild:
        cache = pd.read_csv(cache_path, dtype=str, keep_default_na=False)
        df = df.merge(cache[["uid", "text_hash", "lemmas"]], on=["uid", "text_hash"], how="left")
    else:
        df["lemmas"] = pd.Series([None] * len(df), index=df.index, dtype="object")
    df["lemmas"] = df["lemmas"].astype("object")
    todo = df["lemmas"].isna()
    print(f"{int((~todo).sum()):,} entries from cache, {int(todo.sum()):,} to process")
    if todo.any():
        df.loc[todo, "lemmas"] = lemmatise_texts(df.loc[todo, "text_clean"].tolist(), nlp)
        df[["uid", "text_hash", "lemmas"]].to_csv(cache_path, index=False)
    df["lemmas"] = df["lemmas"].fillna("")
    df["n_tokens"] = df["lemmas"].str.split().str.len()
    return df

In [9]:
nlp = load_nlp(SPACY_MODEL)
book_df = add_lemmas(book_df, nlp, CACHE_DIR / f"lemmas_{SPACY_MODEL}.csv", rebuild=REBUILD_LEMMA_CACHE)
print(f"Median content words per entry: {int(book_df['n_tokens'].median())}")

Loaded spaCy model: en_core_web_sm
0 entries from cache, 69 to process


spaCy:   0%|          | 0/69 [00:00<?, ?it/s]

Median content words per entry: 45


## 5. Segments → TF-IDF

In [10]:
# Diary entries are often 50-200 words, which is too short for stable topic estimates.
# So models are FIT on "segments": consecutive entries (never crossing a chapter or book
# boundary) merged until they reach MIN_SEGMENT_TOKENS content words. Every individual
# entry is then scored against the fitted model afterwards, so you still get per-entry
# topic weights that join back to your knowledge graph on entry_id.

def build_segments(df, min_tokens):
    segs = []
    for (book, chap), g in df.groupby(["book", "chapter_id"], sort=False):
        cur, n = [], 0
        for r in g.itertuples():
            cur.append(r); n += r.n_tokens
            if n >= min_tokens:
                segs.append(cur); cur, n = [], 0
        if cur:
            if segs and segs[-1][0].book == book and segs[-1][0].chapter_id == chap:
                segs[-1].extend(cur)          # fold a short tail into the previous segment
            else:
                segs.append(cur)              # chapter too short to reach the threshold
    rows = []
    for i, s in enumerate(segs):
        yrs = [r.year for r in s if not pd.isna(r.year)]
        rows.append({
            "seg_id": i,
            "book": s[0].book,
            "chapter_id": s[0].chapter_id,
            "first_pos": s[0].pos,
            "uids": [r.uid for r in s],
            "n_entries": len(s),
            "n_tokens": sum(r.n_tokens for r in s),
            "year": float(np.median(yrs)) if yrs else np.nan,
            "lemmas": " ".join(r.lemmas for r in s),
            "text_clean": " ".join(r.text_clean for r in s),
        })
    return pd.DataFrame(rows)

In [11]:
seg_df = build_segments(book_df, MIN_SEGMENT_TOKENS)
print(f"{len(seg_df)} segments (median {int(seg_df['n_tokens'].median())} content words, {seg_df['n_entries'].median():.0f} entries each)")

vec = TfidfVectorizer(lowercase=False, tokenizer=str.split, token_pattern=None, preprocessor=None,
                      ngram_range=NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF, sublinear_tf=True)
X = vec.fit_transform(seg_df["lemmas"])
names = np.array(vec.get_feature_names_out())
keep_rows = X.getnnz(axis=1) > 0
seg_df, X = seg_df[keep_rows].reset_index(drop=True), X[keep_rows]
B = (X > 0).astype(np.float32).tocsr()
print(f"Model matrix: {X.shape[0]} segments x {X.shape[1]} terms")

31 segments (median 137 content words, 2 entries each)
Model matrix: 31 segments x 805 terms


## 6. Choose the number of topics

NPMI coherence (do a topic's top words really co-occur?) and topic diversity (do topics avoid repeating each other?) for each candidate K. In one book, **small K is usually right**: 4–8 topics is often the most interpretable. If the best-coherence K merges things you know are distinct, set `K_FINAL`.

In [12]:
def display_terms(row, names, n=8):
    # Top terms for a topic, skipping n-grams that repeat a word already chosen
    # (so "fever" and "fever shed" do not both crowd the label).
    chosen, words = [], set()
    for i in np.argsort(row)[::-1]:
        w = set(names[i].split())
        if w & words:
            continue
        chosen.append(i); words |= w
        if len(chosen) == n:
            break
    return chosen

def npmi_coherence(H, B, names, topn=10):
    # NPMI coherence: do a topic's top words actually co-occur in the same segments?
    # Range -1..1, higher is better. Pairs that share a word (unigram vs. its own bigram)
    # are skipped, otherwise they inflate the score.
    n_docs = B.shape[0]
    df = np.asarray(B.sum(axis=0)).ravel()
    scores = []
    for row in H:
        idx = display_terms(row, names, topn)
        sub = B[:, idx]
        co = (sub.T @ sub).toarray() / n_docs
        p = df[idx] / n_docs
        vals = []
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                pij = co[i, j]
                vals.append(-1.0 if pij <= 0 else np.log(pij / (p[i] * p[j])) / (-np.log(pij)))
        scores.append(np.mean(vals) if vals else 0.0)
    return np.array(scores)

def fit_nmf(X, k):
    m = NMF(n_components=k, init="nndsvda", random_state=SEED, max_iter=500, tol=1e-4)
    W = m.fit_transform(X)
    return m, W

def row_normalise(M):
    s = M.sum(axis=1, keepdims=True)
    out = np.divide(M, s, out=np.full_like(M, np.nan, dtype=float), where=s > 0)
    return out

PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
def topic_color(t):
    return PALETTE[t % len(PALETTE)]

def show(fig, name):
    fig.write_html(FIG_DIR / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

def plot_topic_words(H, names, labels, n_words=8, ncols=4, title="Top terms per topic"):
    K = H.shape[0]
    nrows = ceil(K / ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[labels[t][:42] for t in range(K)],
                        horizontal_spacing=0.10, vertical_spacing=0.30 / max(nrows, 1))
    for t in range(K):
        idx = display_terms(H[t], names, n_words)
        fig.add_trace(go.Bar(x=H[t, idx][::-1], y=[names[i] for i in idx][::-1], orientation="h",
                             marker_color=topic_color(t), showlegend=False,
                             hovertemplate="%{y}<extra></extra>"),
                      row=t // ncols + 1, col=t % ncols + 1)
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(tickfont=dict(size=10))
    fig.update_annotations(font_size=11)
    fig.update_layout(title=title, height=max(320, 230 * nrows), margin=dict(t=80, b=20))
    return fig

def cluster_order(M, metric="euclidean"):
    if len(M) < 3:
        return list(range(len(M)))
    Z = linkage(pdist(M, metric=metric), method="average", optimal_ordering=True)
    return list(leaves_list(Z))

In [13]:
sweep = []
ks = [k for k in K_RANGE if k < min(X.shape) and k * MIN_SEGS_PER_TOPIC <= X.shape[0]]
if not ks:
    ks = [max(2, min(3, min(X.shape) - 1))]
    print(f"!! Only {X.shape[0]} segments: trying K = {ks[0]} only. Lower MIN_SEGMENT_TOKENS to get more segments.")
for k in ks:
    m, _ = fit_nmf(X, k)
    coh = npmi_coherence(m.components_, B, names, COHERENCE_TOPN)
    tops = [tuple(names[display_terms(r, names, COHERENCE_TOPN)]) for r in m.components_]
    diversity = len({w for t in tops for w in t}) / max(1, sum(len(t) for t in tops))
    sweep.append({"k": k, "coherence_mean": coh.mean(), "coherence_min": coh.min(), "diversity": diversity})
sweep = pd.DataFrame(sweep)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.coherence_mean, name="mean NPMI coherence", mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.coherence_min, name="worst topic coherence", mode="lines+markers", line=dict(dash="dot")), secondary_y=False)
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.diversity, name="topic diversity", mode="lines+markers"), secondary_y=True)
fig.update_layout(title=f"{TITLE}: choosing K", xaxis_title="number of topics (K)", height=400)
fig.update_yaxes(title_text="coherence (NPMI)", secondary_y=False)
fig.update_yaxes(title_text="diversity", range=[0, 1.05], secondary_y=True)
show(fig, "01_k_sweep")

K = int(K_FINAL) if K_FINAL else int(sweep.loc[sweep["coherence_mean"].idxmax(), "k"])
print(f"Using K = {K}" + ("  (set manually)" if K_FINAL else "  (best mean coherence)"))
sweep.round(3)

Using K = 7  (best mean coherence)


,k,coherence_mean,coherence_min,diversity
0,3,0.344,0.240,0.967
1,4,0.345,0.257,0.975
2,5,0.359,0.256,0.940
3,6,0.397,0.234,0.950
4,7,0.409,0.291,0.971


## 7. Fit the model & score every entry

The model is fitted on segments, then each *individual entry* is projected onto the topics, so the time-series below have one point per diary entry. An entry with no usable vocabulary (very short, or all names) gets no score (shown as a gap, `dominant_topic = -1`).

In [14]:
nmf, W = fit_nmf(X, K)
H = nmf.components_
X_e = normalize(vec.transform(book_df["lemmas"]))
We = row_normalise(nmf.transform(X_e))                    # entries x topics, rows sum to 1 (NaN = no signal)
n_ent = len(book_df)
no_signal = np.isnan(We).all(axis=1)
print(f"{int(no_signal.sum())} of {n_ent} entries have no usable vocabulary")

topic_cols = [f"topic_{t}" for t in range(K)]
entry_topics = book_df[["uid", "book", "chapter_id", "entry_id", "pos", "date", "year", "month", "day", "n_tokens"]].copy()
entry_topics[topic_cols] = We
entry_topics["dominant_topic"] = np.where(no_signal, -1, np.nanargmax(np.nan_to_num(We, nan=-1), axis=1))

0 of 69 entries have no usable vocabulary


## 8. Name the topics

Auto-labels are the top three distinct terms. Exemplars are the highest-scoring entries per topic (entries with at least 25 content words, so a two-line entry cannot win on a lucky word). Set `USE_LLM_LABELS = True` for Claude-suggested labels, descriptions and a `kind` (theme / event / setting / artefact); treat them as drafts to review.

In [15]:
top_terms = {t: [names[i] for i in display_terms(H[t], names, 12)] for t in range(K)}
eligible = (book_df["n_tokens"].values >= 25) & ~no_signal

exemplar_rows = []
for t in range(K):
    score = np.where(eligible, np.nan_to_num(We[:, t], nan=-1), -1)
    for rank, i in enumerate(np.argsort(score)[::-1][:5], 1):
        if score[i] < 0:
            break
        r = book_df.loc[i]
        exemplar_rows.append({"topic": t, "rank": rank, "weight": round(float(We[i, t]), 3), "entry_id": r["entry_id"],
                              "chapter_id": r["chapter_id"], "date": r["date"], "text": r["text_clean"][:700]})
exemplars = pd.DataFrame(exemplar_rows)

topic_info = pd.DataFrame({"topic": range(K)})
topic_info["auto_label"] = [" / ".join(top_terms[t][:3]) for t in range(K)]
topic_info["label"] = topic_info["auto_label"]
topic_info["description"] = ""
topic_info["kind"] = ""

if USE_LLM_LABELS:
    payload = [{"topic": t, "top_terms": top_terms[t],
                "exemplars": [{"entry_id": r.entry_id, "date": r.date, "text": r.text[:400]}
                              for r in exemplars[exemplars.topic == t].head(3).itertuples()]}
               for t in range(K)]
    note = f"All topics come from one book: {TITLE}. " + CONTEXT_NOTE
    for r in llm_label_topics(payload, note.strip()):
        topic_info.loc[topic_info.topic == r["topic"], ["label", "description", "kind"]] = [r["label"], r["description"], r["kind"]]

topic_label = {t: f"T{t}: {topic_info.loc[t, 'label']}" for t in range(K)}
topic_info

,topic,auto_label,label,description,kind
0,0,captain / hospital / ill,captain / hospital / ill,,
1,1,run / thief / orphan,run / thief / orphan,,
2,2,cook / oatmeal / rude,cook / oatmeal / rude,,
3,3,brother / rush / visit,brother / rush / visit,,
4,4,storm / wind / hatch,storm / wind / hatch,,
5,5,pot / kitchen / scrub,pot / kitchen / scrub,,
6,6,steamer / eat / milk,steamer / eat / milk,,


## 9. Visualisations

All figures are interactive and saved as standalone HTML in `output/topics_book/<book>/figures/`.

### 9a. What each topic is made of

In [16]:
fig = plot_topic_words(H, names, [topic_label[t] for t in range(K)], n_words=8, ncols=min(4, K),
                       title=f"{TITLE}: top terms per topic")
show(fig, "02_topic_terms")

### 9b. Themes across the narrative

Prepare the time axes once: smoothed topic shares (rolling mean over `SMOOTH_WINDOW` entries, so single noisy entries do not dominate) and chapter boundaries.

In [17]:
Ws = pd.DataFrame(We).rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().ffill().bfill()
Ws = Ws.div(Ws.sum(axis=1), axis=0).fillna(0)
Ps = Ws.values                                           # smoothed entries x topics
dom_smooth = Ps.argmax(axis=1)

xpos = book_df["pos"].values + 1                         # 1-based entry number along the narrative
chap_start = book_df.groupby("chapter_id", sort=False)["pos"].min() + 1
def chapter_axis(fig, rows=None):
    fig.update_xaxes(tickmode="array", tickvals=list(chap_start.values), ticktext=list(chap_start.index),
                     tickangle=-45, title_text="narrative position (chapter starts marked)")
    for x in chap_start.values[1:]:
        fig.add_vline(x=x - 0.5, line_width=0.5, line_dash="dot", line_color="grey")
hover_meta = np.stack([book_df["entry_id"].values, book_df["date"].values], axis=1)

**Stacked area** — each band is a topic's share of the diary at that point (smoothed). The overall shape of the book's thematic arc:

In [18]:
fig = go.Figure()
for t in range(K):
    fig.add_trace(go.Scatter(x=xpos, y=Ps[:, t], stackgroup="one", name=topic_label[t][:46], line=dict(width=0.4),
                             fillcolor=topic_color(t), line_color=topic_color(t), customdata=hover_meta,
                             hovertemplate="%{customdata[0]} (%{customdata[1]})<br>" + topic_label[t][:46] + ": %{y:.0%}<extra></extra>"))
fig.update_layout(title=f"{TITLE}: topic share across the narrative (smoothed, {SMOOTH_WINDOW}-entry window)",
                  yaxis=dict(title="share", tickformat=".0%", range=[0, 1]), height=520, hovermode="x unified",
                  legend=dict(orientation="v", x=1.02, y=1))
chapter_axis(fig)
show(fig, "03_stacked_area_narrative")

**Heatmap, entry by entry** — unsmoothed, so you can see the raw texture the smoothed view averages away (gaps = entries with no usable vocabulary).

In [19]:
fig = go.Figure(go.Heatmap(z=We.T, x=xpos, y=[topic_label[t][:46] for t in range(K)], colorscale="Viridis",
                           customdata=np.tile(hover_meta[:, 0], (K, 1)), colorbar=dict(title="weight"),
                           hovertemplate="%{customdata}<br>%{y}: %{z:.0%}<extra></extra>"))
fig.update_layout(title=f"{TITLE}: topic weight in each entry", height=max(350, 40 * K + 180))
fig.update_yaxes(autorange="reversed")
chapter_axis(fig)
show(fig, "04_heatmap_entries")

**Dominant-topic ribbon** — which topic leads at each point (smoothed). Long same-colour runs are sustained phases; frequent colour changes are a book that moves quickly between concerns.

In [20]:
fig = go.Figure()
for t in range(K):
    mask = dom_smooth == t
    if not mask.any():
        continue
    fig.add_trace(go.Bar(x=xpos[mask], y=np.ones(mask.sum()), name=topic_label[t][:46], marker_color=topic_color(t),
                         customdata=hover_meta[mask], width=1,
                         hovertemplate="%{customdata[0]} (%{customdata[1]})<extra>" + topic_label[t][:46] + "</extra>"))
fig.update_layout(title=f"{TITLE}: leading topic through the narrative", barmode="overlay", bargap=0, height=300,
                  yaxis=dict(visible=False), legend=dict(orientation="v", x=1.02, y=1))
chapter_axis(fig)
show(fig, "05_dominant_ribbon")

**Chapter × topic** — the same story at chapter resolution, weighted by how much text each entry contains.

In [21]:
Wz = np.nan_to_num(We) * (book_df["n_tokens"].values[:, None] * (~no_signal)[:, None])
wt = (book_df["n_tokens"].values * (~no_signal))
chap_ids = list(chap_start.index)
ch_rows = []
for c in chap_ids:
    idx = (book_df["chapter_id"] == c).values
    ch_rows.append(Wz[idx].sum(axis=0) / max(wt[idx].sum(), 1))
chapter_topics = pd.DataFrame(ch_rows, index=chap_ids, columns=[topic_label[t] for t in range(K)])

fig = px.imshow(chapter_topics.T.values, x=chap_ids, y=[topic_label[t][:46] for t in range(K)], aspect="auto",
                color_continuous_scale="Viridis", text_auto=".0%" if len(chap_ids) * K <= 200 else False,
                labels=dict(color="share"), title=f"{TITLE}: topic share by chapter")
fig.update_layout(height=max(350, 40 * K + 180))
show(fig, "06_chapter_by_topic")

### 9c. Where does the story turn?

For each position in the book, compare the average topic mix of the `SHIFT_WINDOW` entries **before** with the `SHIFT_WINDOW` entries **after** (Jensen–Shannon distance: 0 = identical mix, 1 = completely different). Peaks are candidate narrative turning points — a change of place, a death, an arrival. The table below the plot names the topic handing over to the next and quotes the entry, so you can check each against what actually happens in the book.

In [22]:
Wf = pd.DataFrame(We).ffill().bfill().fillna(0).values
w = SHIFT_WINDOW
jsd = np.full(n_ent, np.nan)
before_dom, after_dom = np.full(n_ent, -1), np.full(n_ent, -1)
for i in range(w, n_ent - w + 1):
    b, a = Wf[i - w:i].mean(axis=0), Wf[i:i + w].mean(axis=0)
    if b.sum() > 0 and a.sum() > 0:
        jsd[i] = jensenshannon(b / b.sum(), a / a.sum(), base=2)
        before_dom[i], after_dom[i] = b.argmax(), a.argmax()

shifts = pd.DataFrame()
if np.isfinite(jsd).sum() > 3:
    peaks, _ = find_peaks(np.nan_to_num(jsd), distance=max(2, w), prominence=0.5 * np.nanstd(jsd))
    peaks = sorted(peaks, key=lambda i: -jsd[i])[:N_SHIFTS]
    shifts = pd.DataFrame([{
        "entry_number": i + 1, "entry_id": book_df.loc[i, "entry_id"], "chapter_id": book_df.loc[i, "chapter_id"],
        "date": book_df.loc[i, "date"], "shift_strength": round(float(jsd[i]), 3),
        "from_topic": topic_label[before_dom[i]], "to_topic": topic_label[after_dom[i]],
        "entry_starts": book_df.loc[i, "text_clean"][:170] + "…"} for i in peaks]).sort_values("entry_number")

    fig = go.Figure(go.Scatter(x=xpos, y=jsd, mode="lines", line=dict(color="#444"), name="topic-mix change",
                               customdata=hover_meta, hovertemplate="%{customdata[0]} (%{customdata[1]})<br>change: %{y:.2f}<extra></extra>"))
    if len(shifts):
        fig.add_trace(go.Scatter(x=shifts["entry_number"], y=shifts["shift_strength"], mode="markers+text",
                                 marker=dict(size=11, color="#d62728"), text=shifts["entry_id"].str.extract(r"(e\d+)$")[0].fillna(""),
                                 textposition="top center", name="largest shifts",
                                 hovertext=shifts["from_topic"].str[:30] + " → " + shifts["to_topic"].str[:30]))
    fig.update_layout(title=f"{TITLE}: where the topic mix changes most", yaxis_title="Jensen–Shannon distance (before vs. after)", height=420)
    chapter_axis(fig)
    show(fig, "07_topic_shifts")
else:
    print("Book too short for shift detection with this SHIFT_WINDOW.")
shifts

,entry_number,entry_id,chapter_id,date,shift_strength,from_topic,to_topic,entry_starts
2,6,chc02_e5,c02,"May 8, 1847",0.764,T6: steamer / eat / milk,T2: cook / oatmeal / rude,"As we waited to board, I heard someone say tha..."
5,15,chc02_e14,c02,"May 19, 1847",0.679,T2: cook / oatmeal / rude,T4: storm / wind / hatch,"It has been days since I have written, for no ..."
6,20,chc02_e19,c02,"May 24, 1847",0.468,T2: cook / oatmeal / rude,T4: storm / wind / hatch,Mr. Tattersall is in a stupor. He lies motionl...
4,28,chc03_e5,c03,"June 9, 1847",0.698,T4: storm / wind / hatch,T6: steamer / eat / milk,"Our Patrick is gone. He died so quietly, poor ..."
7,39,chc03_e16,c03,"June 22, 1847",0.397,T0: captain / hospital / ill,T0: captain / hospital / ill,I was so worried about Ma and afraid of what m...
3,45,chc04_e1,c04,"July 3, 1847",0.760,T0: captain / hospital / ill,T1: run / thief / orphan,"I am writing this in Montreal, and my heart st..."
0,53,chc04_e9,c04,"July 16, 1847",0.823,T1: run / thief / orphan,T3: brother / rush / visit,I waited all day for Michael to visit and at t...
1,59,chc04_e15,c04,"July 24, 1847",0.777,T3: brother / rush / visit,T5: pot / kitchen / scrub,A man came for me in a wagon and drove me thro...


### 9d. Each topic's arc: when does it live?

One row per topic. The dot is the topic's **centre of mass** along the narrative (0 = first entry, 1 = last) and the bar is ± one standard deviation of its weight: a short bar means a topic concentrated in one phase of the book, a long bar means it recurs throughout. (A topic spread perfectly evenly would have a standard deviation of about 0.29.)

In [23]:
posn = book_df["pos"].values / max(1, n_ent - 1)
mass = Wz.sum(axis=0)
centroid = (Wz * posn[:, None]).sum(axis=0) / mass
spread = np.sqrt((Wz * (posn[:, None] - centroid) ** 2).sum(axis=0) / mass)

def arc_label(c, s):
    if s > 0.24:
        return "recurs throughout"
    return "early" if c < 0.33 else ("middle" if c < 0.67 else "late")

peak_i = Ps.argmax(axis=0)
topic_summary = pd.DataFrame({
    "topic": range(K), "label": [topic_label[t] for t in range(K)], "kind": topic_info["kind"],
    "top_terms": [", ".join(top_terms[t][:8]) for t in range(K)],
    "share_of_book": mass / mass.sum(), "centroid": centroid, "spread": spread,
    "arc": [arc_label(c, s) for c, s in zip(centroid, spread)],
    "peak_entry": book_df.loc[peak_i, "entry_id"].values, "peak_date": book_df.loc[peak_i, "date"].values,
    "peak_share": Ps[peak_i, np.arange(K)],
})
order = np.argsort(centroid)
fig = go.Figure(go.Scatter(
    x=centroid[order], y=[topic_label[t][:50] for t in order], mode="markers",
    marker=dict(size=12, color=[topic_color(t) for t in order]),
    error_x=dict(type="data", array=spread[order], color="#888", thickness=2),
    customdata=np.stack([topic_summary.loc[order, "arc"], topic_summary.loc[order, "peak_entry"], topic_summary.loc[order, "share_of_book"]], axis=1),
    hovertemplate="%{y}<br>%{customdata[0]}; peak at %{customdata[1]}<br>share of book: %{customdata[2]:.0%}<extra></extra>"))
fig.update_layout(title=f"{TITLE}: when each topic lives (dot = centre of mass, bar = ±1 SD)",
                  xaxis=dict(title="narrative position (0 = start, 1 = end)", range=[-0.3, 1.3]), height=max(320, 55 * K + 150))
fig.add_vrect(x0=0, x1=1, fillcolor="lightgrey", opacity=0.15, line_width=0)
yaxis_order = [topic_label[t][:50] for t in order]
fig.update_yaxes(categoryorder="array", categoryarray=yaxis_order[::-1])
show(fig, "08_topic_arcs")
topic_summary.drop(columns=["label"]).round(3)

,topic,kind,top_terms,share_of_book,centroid,spread,arc,peak_entry,peak_date,peak_share
0,0,,"captain, hospital, ill, ship, island, doctor, ...",0.123,0.526,0.144,middle,chc03_e18,"June 24, 1847",0.859
1,1,,"run, thief, orphan, disappear, boy, foot, shed...",0.164,0.670,0.174,late,chc04_e5,"July 12, 1847",0.668
2,2,,"cook, oatmeal, rude, ration, fireplace, girl, ...",0.088,0.201,0.193,early,chc02_e11,"May 14, 1847",0.703
3,3,,"brother, rush, visit, beg, employer, shed, new...",0.113,0.661,0.196,middle,chc04_e11,"July 18, 1847",0.825
4,4,,"storm, wind, hatch, wail, rest, howl, air, toss",0.161,0.361,0.229,middle,chc02_e20,"May 27, 1847",0.608
5,5,,"pot, kitchen, scrub, fairy, set, work, stair, ...",0.221,0.907,0.128,late,chc05_e4,"August 5, 1847",0.906
6,6,,"steamer, eat, milk, liverpool, canada, hope, e...",0.130,0.270,0.339,recurs throughout,chc01_e1,"April 30, 1847",0.959


### 9e. Calendar time

Same topic shares, but placed by the date in the diary instead of entry order. The top panel shows how many entries fall in each month: where it is empty, the diary skips time, and where it is tall, the narrative dwells. Months without any dated entry are left blank, not interpolated. Skipped automatically if too few entries have a readable month.

In [24]:
calendar_ok = n_dated >= 0.5 * n_ent and n_dated >= 8
if not calendar_ok:
    print("Fewer than half the entries have a readable year+month: skipping calendar view. "
          "Check the 'date' column or set START_YEAR in the config.")
else:
    dfc = pd.DataFrame(We, columns=range(K))
    dfc["ym"] = book_df["ym"].values
    monthly = dfc.dropna(subset=["ym"]).groupby("ym").mean()
    counts = dfc.dropna(subset=["ym"]).groupby("ym").size()
    full = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
    if len(full) > 400:
        print("Date span is very long (>400 months); calendar view skipped.")
    else:
        monthly = monthly.reindex(full)
        counts = counts.reindex(full).fillna(0)
        monthly_f = monthly.div(monthly.sum(axis=1), axis=0).fillna(0)
        xs = [str(p) for p in full]
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.22, 0.78], vertical_spacing=0.04)
        fig.add_trace(go.Bar(x=xs, y=counts.values, marker_color="#888", name="entries per month", showlegend=False), row=1, col=1)
        for t in range(K):
            fig.add_trace(go.Bar(x=xs, y=monthly_f[t].values, name=topic_label[t][:46], marker_color=topic_color(t)), row=2, col=1)
        fig.update_layout(barmode="stack", bargap=0.05, height=560, title=f"{TITLE}: topics in calendar time",
                          legend=dict(orientation="v", x=1.02, y=0.8))
        fig.update_yaxes(title_text="entries", row=1, col=1)
        fig.update_yaxes(title_text="topic share", tickformat=".0%", row=2, col=1)
        fig.update_xaxes(tickangle=-45, row=2, col=1)
        show(fig, "09_calendar_time")

## 10. Export

In [25]:
topic_summary.to_csv(OUT_DIR / "topic_summary.csv", index=False)
pd.DataFrame([{"topic": t, "rank": r + 1, "term": names[i], "weight": H[t, i]}
              for t in range(K) for r, i in enumerate(display_terms(H[t], names, 25))]).to_csv(OUT_DIR / "topic_terms.csv", index=False)
entry_topics.to_csv(OUT_DIR / "entry_topics.csv", index=False)
chapter_topics.to_csv(OUT_DIR / "chapter_topics.csv")
exemplars.to_csv(OUT_DIR / "exemplars.csv", index=False)
topic_info.to_csv(OUT_DIR / "topic_labels.csv", index=False)
if len(shifts):
    shifts.to_csv(OUT_DIR / "topic_shifts.csv", index=False)
print("Wrote to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print("  ", f.name)

Wrote to /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/output/topics_book/SofS_entries_halved
   chapter_topics.csv
   entry_topics.csv
   exemplars.csv
   topic_labels.csv
   topic_shifts.csv
   topic_summary.csv
   topic_terms.csv


## 11. Reading the results (and what to check)

1. **Read `exemplars.csv` first.** The five highest-scoring entries per topic are the ground truth for what a topic means; the label is only shorthand.
2. **Check the shift table against the book.** A detected shift is a change in *vocabulary*, not necessarily in *plot*. Most will line up with real turns (a move, a death, a new setting); some will be a change of register or of who the narrator is talking about. Both are findings, but only a reader can tell which.
3. **Set expectations for K.** A single book gives a few hundred entries at most. If two topics look like one idea split in two (adjacent colours in the ribbon that always rise and fall together), lower `K_FINAL`; if one topic mixes two ideas you know are distinct, raise it.
4. **Dominant ≠ most of the text.** The ribbon shows the *leading* topic; the stacked area shows how close the others are. A phase where the leader is only 30% is a mixed phase, not a pure one.
5. **Entries without vocabulary** leave gaps (`dominant_topic = -1`). Many gaps clustered in one chapter usually mean short, dialogue-heavy entries, or names dominating the text.
6. **Joining to the KG:** `entry_topics.csv` has `book` and `entry_id`, so topic weights can attach to `Entry` nodes, and the shift entries can be cross-checked against your `Stay`, `HealthState` and `Incident` nodes (e.g. does the "illness" topic really climb where `HealthState` nodes appear?).